In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 4x4 Systolic Array in Psyre (`bf16` Inputs, `fp32` MAC, Weight- & Output-Stationary)

This notebook illustrates the 4x4 systolic array implemented in `systolic.py` and renders interactive Stage-1 `SYNC` schematics (`visualize.visualize_graph`):

- **Distributed Per-PE Control + Minimal Global Orchestrator**:
  - Minimal 3-state global orchestrator `automaton(name='mode_ctrl')` in `systolic_control` sequencing global WS weight-load (4 cycles), WS compute (`reload_weights`), and OS mode switching (`switch_mode`), broadcasting `GlobalControl(mode, load_weights, datapath_reset)`.
  - Per-PE local 1-register `automaton(name='pe_ctrl')` attached to each `systolic_pe`, cycling through `PEState` (`IDLE`, `WS_ACTIVE`, `OS_INIT`, `OS_ACC1`, `OS_ACC2`, `OS_ACC3`) with zero `Stream.once` counters and zero `pnr` reset registers; its state register also acts as the registered 1-bit wavefront token (`wave_out = state != PEState.IDLE`) passed East and South between neighbors.
- **Processing Element (`systolic_pe`) & 4x4 Mesh (`systolic_array`)**:
  - Only **4 registers per PE** (`row_reg`, `col_reg`, merged `fp32` `acc_reg`, and `pe_ctrl/pns`).
  - Accepts IEEE-754 `bf16` inputs (converted to `bf16` before being provided as inputs to the graph), multiplies in `fp32`, and merges the PE's stationary weight and accumulator into a single `fp32` register (`acc_reg`).
  - Uses `delay`-based boundary `skew_streams` and `deskew_streams` sharing a single saturated counter across lanes, and supports a global single-cycle synchronous reset (`reset_stream`).
- **Interactive Stage-1 Visualization**:
  - Automaton scopes (`mode_ctrl` and each PE's `pe_i_j/pe_ctrl`) start collapsed as interactive summary boxes (`⊞` / `⊟`) that expand and collapse when clicked.
  - Only feedback back-edges are rendered dashed; forward `fby` pipeline inputs stay solid.

In [ ]:
# @title Psyre imports

import logging
# !pip install -q psyre

# pylint: disable=g-importing-member,g-multiple-import
from psyre.builder import build_graph
from psyre.common import (
    counter,
    ifthenelse,
    logger,
    reset,
)
from examples.systolic import (
    ARRAY_SIZE,
    PEState,
    SystolicMode,
    build_os_schedule,
    build_ws_schedule,
    deskew_streams,
    extract_matrices,
    extract_os_pe_matrices,
    reference_matmul_bf16_fp32,
    skew_streams,
    systolic_array,
    systolic_control,
    systolic_pe,
    to_bf16,
    to_fp32,
)
from psyre.scheduler import Scheduler
from psyre.stream import Stream
from psyre.visualize import visualize_graph

# pylint: enable=g-importing-member,g-multiple-import

logger.setLevel(logging.WARNING)

## 1. Building Blocks: Single PE (4 Registers), Skew/Deskew, and Global Orchestrator

Before inspecting the full 4x4 mesh, we visualize the three core building blocks as lowered Stage-1 `SYNC` primitive graphs:
1. **Single Processing Element (`systolic_pe`)**: local 1-register control automaton (`pe_ctrl`, emitting `wave_out = state != PEState.IDLE`), `bf16` inputs, merged `fp32` stationary weight and accumulator register (`acc_reg`), `*` and `+` MAC operators, direct-store vs. accumulate multiplexer (`acc_mux`), and synchronous reset on pipeline registers (`row_reg`, `col_reg`, `acc_reg`) — 4 `fby` registers total.
2. **Boundary Skew & Deskew (`skew_streams`, `deskew_streams`)**: built on `delay` with a single shared saturated counter compared against constant lane offsets (`1`, `2`, `3`) to align diagonal wavefronts across the 4 rows and 4 columns.
3. **Global Phase Orchestrator (`systolic_control`)**: minimal 3-state automaton (`mode_ctrl`) sequencing WS weight-load, WS compute, and OS mode. Click the `⊞` automaton box in the viewer to expand it.

In [ ]:
# @title 1.1 Stage-1 schematic: Single Systolic Processing Element (PE)


def single_pe_circuit():
  """Constructs a single 4-register systolic PE with its local control FSM."""
  row_in = Stream(to_bf16([2.0, 4.0, 1.5, 2.0]), name='row_in')
  col_in = Stream(to_fp32([3.0, 5.0, 2.0, 0.5]), name='col_in')
  mode = Stream(
      [
          SystolicMode.WEIGHT_STATIONARY,
          SystolicMode.WEIGHT_STATIONARY,
          SystolicMode.OUTPUT_STATIONARY,
          SystolicMode.OUTPUT_STATIONARY,
      ],
      name='mode',
  )
  load_w = Stream([True, False, False, False], name='load_weights')
  accumulate = Stream([False, True, False, True], name='accumulate')
  dp_reset = Stream([False, False, True, False], name='datapath_reset')
  pe = systolic_pe(
      row_in,
      col_in,
      mode=mode,
      load_weights=load_w,
      datapath_reset=dp_reset,
      accumulate=accumulate,
      name='pe_0_0',
  )
  return (
      pe.row_out,
      pe.col_out,
      pe.ps_out,
      pe.acc,
      pe.state,
      pe.valid,
      pe.wave_out,
  )


g_pe = build_graph(single_pe_circuit)
visualize_graph(
    g_pe,
    title='Stage-1 SYNC Single Systolic PE (4 registers: pe_ctrl + row/col/acc)',
    height_px=520,
)

In [ ]:
# @title 1.2 Stage-1 schematic: Delay-based Skew and Deskew Boundary Network


def skew_deskew_circuit():
  """Constructs a 4-lane skew + deskew pipeline sharing one tick counter."""
  lanes = [
      Stream(
          to_bf16([10.0, 20.0, 30.0, 40.0, 0.0, 0.0, 0.0]),
          name=f'in_{j}',
      )
      for j in range(ARRAY_SIZE)
  ]
  tick = counter(max_val=ARRAY_SIZE - 1, name='tick')
  skewed = skew_streams(lanes, tick=tick, prefix='skew')
  valid = tick >= (ARRAY_SIZE - 1)
  return deskew_streams(skewed, valid=valid, prefix='deskew')


g_skew = build_graph(skew_deskew_circuit)
visualize_graph(
    g_skew,
    title='Stage-1 SYNC 4-Lane Boundary Skew & Deskew (shared counter + delay)',
    height_px=520,
)

In [ ]:
# @title 1.3 Stage-1 schematic: Global Phase Orchestrator (mode_ctrl)


def global_control_circuit():
  """Constructs the minimal global phase-orchestration automaton."""
  switch_mode = Stream(
      [False, False, False, False, False, True, False, False, False, False],
      name='switch_mode',
  )
  reload_weights = Stream([False] * 10, name='reload_weights')
  ctrl = systolic_control(
      initial_mode=SystolicMode.WEIGHT_STATIONARY,
      switch_mode=switch_mode,
      reload_weights=reload_weights,
  )
  return ctrl.mode, ctrl.load_weights, ctrl.datapath_reset


g_ctrl = build_graph(global_control_circuit)
visualize_graph(
    g_ctrl,
    title=(
        'Stage-1 SYNC Global Phase Orchestrator '
        '(click ⊞ mode_ctrl to expand WS_LOAD / WS_COMPUTE / OS)'
    ),
    height_px=520,
)

## 2. Weight-Stationary Mode: Pipelined 4x4 Matrix Products & Weight Reloading

In `SystolicMode.WEIGHT_STATIONARY`, the global orchestrator `mode_ctrl` and per-PE `pe_ctrl` execute:
1. Global weight load (`load_weights=True`, 4 ticks): shifts weight columns $W[3 \dots 0, j]$ from North to South so each PE $(k, j)$ latches its `bf16` weight $W_{k, j}$ into its merged `fp32` accumulator register (`acc_reg`) while `pe_ctrl` rests in `PEState.IDLE`.
2. Streaming compute (`load_weights=False`): streams skewed 4x4 activation matrices $A^{(0)}, A^{(1)}, \dots$ from West to East back-to-back every 4 cycles. Each PE transitions `PEState.IDLE -> PEState.WS_ACTIVE` as the local 1-bit wavefront arrives, forwarding `wave_out = state != PEState.IDLE` East and South. Row 0 PEs compute $A^{(m)}_{i, 0} \cdot W_{0, j}$ directly, while rows $1 \dots 3$ accumulate partial sums flowing South through `col_reg` in `fp32`. Asserting `reload_weights=True` returns the orchestrator to weight loading and pulses `datapath_reset`.

In [ ]:
# @title 2.1 Run pipelined 4x4 Weight-Stationary products (with weight reload)

weights_epoch0 = to_bf16([
    [1.0, 2.0, -1.0, 0.5],
    [0.5, -1.5, 2.0, 1.0],
    [2.0, 0.25, 1.0, -0.5],
    [-1.0, 1.0, 0.5, 2.0],
])
a0 = to_bf16([
    [1.0, 2.0, 3.0, 4.0],
    [0.5, 1.5, -2.0, 1.0],
    [-1.0, 0.0, 2.5, -0.5],
    [2.0, -1.0, 0.5, 1.5],
])
a1 = to_bf16([
    [1.0 + 2**-7, 2**-5, -1.25, 0.75],
    [2.5, -0.5, 1.125, 3.0],
    [0.25, 1.75, -2.25, 0.5],
    [-1.5, 0.5, 2.0, -0.25],
])
weights_epoch1 = to_bf16([
    [0.5, -0.5, 1.5, 2.0],
    [1.0, 2.0, -1.0, 0.25],
    [-1.5, 0.75, 0.5, 1.0],
    [2.0, 1.0, -0.25, -1.0],
])
a2 = to_bf16([
    [2.0, -1.0, 0.5, 3.0],
    [1.25, 2.5, -1.5, 0.5],
    [-0.5, 1.0, 2.0, -2.0],
    [3.0, 0.25, -1.0, 1.5],
])

ws_epochs = [
    (weights_epoch0, [a0, a1]),
    (weights_epoch1, [a2]),
]
ws_row_seqs, ws_col_seqs, ws_reload_seq, ws_horizon = build_ws_schedule(
    ws_epochs
)


def ws_pipeline_circuit():
  """Runs 2 weight epochs (3 pipelined 4x4 matrix products) in WS mode."""
  row_streams = [
      Stream(ws_row_seqs[i], name=f'ws_row_in_{i}') for i in range(ARRAY_SIZE)
  ]
  skewed_rows = skew_streams(row_streams, prefix='ws_row')
  col_streams = [
      Stream(ws_col_seqs[j], name=f'ws_col_{j}') for j in range(ARRAY_SIZE)
  ]
  out = systolic_array(
      skewed_rows,
      col_streams,
      initial_mode=SystolicMode.WEIGHT_STATIONARY,
      reload_weights=Stream(ws_reload_seq, name='reload_weights'),
  )
  return out.deskewed_outputs, out.accumulators, out.pe_states[0][0]


ws_deskewed_tr, ws_acc_tr, ws_state_tr = Scheduler.run(
    ws_pipeline_circuit, horizon=ws_horizon
)
g_ws = Scheduler.graph

ws_matrices = extract_matrices(ws_deskewed_tr)
ws_expected = [
    reference_matmul_bf16_fp32(a0, weights_epoch0),
    reference_matmul_bf16_fp32(a1, weights_epoch0),
    reference_matmul_bf16_fp32(a2, weights_epoch1),
]
print(f'Total SYNC ticks: {ws_horizon} | Stage-1 nodes: {len(g_ws.nodes)}')
print(
    'PE (0, 0) state timeline:',
    [s.name for s in ws_state_tr],
)
for idx, (mat, ref_mat) in enumerate(zip(ws_matrices, ws_expected)):
  assert mat == ref_mat
  print(f'\nCompleted WS Matrix C^({idx}) (matches fp32 reference):')
  for row in mat:
    print(' ', [round(v, 6) for v in row])

## 3. Output-Stationary Mode, Direct-Store Initialization, and Global Reset

In `SystolicMode.OUTPUT_STATIONARY`, each PE $(i, j)$ computes $C^{(m)}_{i, j} = \sum_{k=0}^3 \text{bf16}(A^{(m)}_{i, k}) \cdot \text{bf16}(B^{(m)}_{k, j})$ in place:
- Each PE's 1-register local automaton `pe_ctrl` starts in `PEState.IDLE`, storing $\text{bf16}(A^{(m)}_{i, 0}) \cdot \text{bf16}(B^{(m)}_{0, j})$ directly into `acc_reg` when its 1-bit wavefront token (`wave_west & wave_north`) arrives at tick $d = i + j$, then cycles through `PEState.OS_ACC1 -> OS_ACC2 -> OS_ACC3` (accumulating $k = 1, 2, 3$ in `fp32` and asserting `valid` in `OS_ACC3`) and `PEState.OS_INIT` (zero-bubble next-tile direct store).
- Because `wave_out = state != PEState.IDLE` stays `True` across `OS_ACC1..3` and `OS_INIT`, diagonal wavefront stagger emerges locally using only the PE's single FSM state register — no separate `wave_reg` or centralized shift chains.
- Below we also demonstrate the global single-cycle synchronous reset (`reset_stream`), which immediately resets the global orchestrator and all 16 PEs mid-computation.

In [ ]:
# @title 3.1 Run pipelined 4x4 Output-Stationary products & Global Single-Cycle Reset

b0 = to_bf16([
    [2.0, -1.0, 0.5, 1.0],
    [0.5, 1.5, -2.0, 0.25],
    [-1.0, 2.0, 1.0, -0.5],
    [1.5, -0.5, 0.75, 2.0],
])
b1 = to_bf16([
    [1.0, -0.5, 2.0, 0.25],
    [2**-5, 1.25, -0.75, 1.5],
    [2**-5, -1.5, 0.5, -2.0],
    [2**-5, 0.75, 1.25, 0.5],
])

os_pairs = [(a0, b0), (a1, b1)]
os_row_seqs, os_col_seqs, os_clean_horizon = build_os_schedule(os_pairs)

# Prepend 3 corrupted ticks, assert global single-cycle reset at tick 3, and
# then execute the 2 pipelined 4x4 OS matrix products back-to-back.
abort_ticks = 3
os_total_horizon = abort_ticks + os_clean_horizon
os_reset_seq = [t == abort_ticks for t in range(os_total_horizon)]
os_full_rows = [
    to_bf16([99.0] * abort_ticks) + os_row_seqs[i] for i in range(ARRAY_SIZE)
]
os_full_cols = [
    to_bf16([77.0] * abort_ticks) + os_col_seqs[j] for j in range(ARRAY_SIZE)
]


def os_pipeline_with_reset_circuit():
  """Runs OS pipelined matrix products after a single-cycle global reset."""
  rst = Stream(os_reset_seq, name='global_reset')
  with reset(rst):
    row_streams = [
        Stream(os_full_rows[i], name=f'os_row_in_{i}')
        for i in range(ARRAY_SIZE)
    ]
    col_streams = [
        Stream(os_full_cols[j], name=f'os_col_in_{j}')
        for j in range(ARRAY_SIZE)
    ]
    os_tick = counter(max_val=ARRAY_SIZE - 1, name='os_skew_cnt')
    skewed_rows = skew_streams(row_streams, tick=os_tick, prefix='os_row')
    skewed_cols = skew_streams(col_streams, tick=os_tick, prefix='os_col')
  out = systolic_array(
      skewed_rows,
      skewed_cols,
      initial_mode=SystolicMode.OUTPUT_STATIONARY,
      reset_stream=rst,
  )
  return (
      out.deskewed_outputs,
      out.accumulators,
      out.pe_valid,
      out.pe_states[0][0],
  )


os_deskewed_tr, os_acc_tr, os_valid_tr, os_state_tr = Scheduler.run(
    os_pipeline_with_reset_circuit,
    horizon=os_total_horizon,
)
g_os = Scheduler.graph

post_rst_deskewed = [col[abort_ticks:] for col in os_deskewed_tr]
post_rst_acc = [
    [os_acc_tr[i][j][abort_ticks:] for j in range(ARRAY_SIZE)]
    for i in range(ARRAY_SIZE)
]
post_rst_valid = [
    [os_valid_tr[i][j][abort_ticks:] for j in range(ARRAY_SIZE)]
    for i in range(ARRAY_SIZE)
]

os_matrices = extract_matrices(post_rst_deskewed)
os_pe_matrices = extract_os_pe_matrices(post_rst_acc, post_rst_valid)
os_expected = [
    reference_matmul_bf16_fp32(a0, b0),
    reference_matmul_bf16_fp32(a1, b1),
]
assert os_matrices == os_expected
assert os_pe_matrices == os_expected

print(
    f'Total SYNC ticks: {os_total_horizon} (global reset at tick {abort_ticks})'
    f' | Stage-1 nodes: {len(g_os.nodes)}'
)
print('PE (0, 0) state timeline:', [s.name for s in os_state_tr])
for idx, mat in enumerate(os_matrices):
  print(f'\nCompleted OS Matrix C^({idx}) (deskewed & in-place PE match):')
  for row in mat:
    print(' ', [round(v, 6) for v in row])

## 4. Top-Level Mode Switching & Combined Schematic

Finally, we run a single execution that transitions the top-level automaton `mode_ctrl` from `SystolicMode.WEIGHT_STATIONARY` to `SystolicMode.OUTPUT_STATIONARY` and visualizes the full combined 4x4 systolic array schematic.

In [ ]:
# @title 4.1 Run WS <-> OS mode switching and render combined Stage-1 schematic

ws_part_rows, ws_part_cols, _, ws_part_h = build_ws_schedule(
    [(weights_epoch0, [a0])]
)
os_part_rows, os_part_cols, os_part_h = build_os_schedule([(a1, b1)])
combined_horizon = ws_part_h + os_part_h
switch_seq = [t == ws_part_h - 1 for t in range(combined_horizon)]


def combined_mode_switch_circuit():
  """Runs a WS 4x4 matrix product followed by an OS 4x4 matrix product."""
  global_tick = counter(name='global_tick')
  is_os_phase = global_tick >= ws_part_h
  with reset(global_tick == ws_part_h):
    skew_tick = counter(max_val=ARRAY_SIZE - 1, name='mix_skew_cnt')
    skewed_ws_rows = skew_streams(
        [
            Stream(row_w + [0.0] * os_part_h, name=f'ws_row_in_{i}')
            for i, row_w in enumerate(ws_part_rows)
        ],
        tick=skew_tick,
        prefix='mix_ws_row',
    )
    skewed_os_rows = skew_streams(
        [
            Stream([0.0] * ws_part_h + row_o, name=f'os_row_in_{i}')
            for i, row_o in enumerate(os_part_rows)
        ],
        tick=skew_tick,
        prefix='mix_os_row',
    )
    skewed_os_cols = skew_streams(
        [
            Stream([0.0] * ws_part_h + col_o, name=f'os_col_in_{j}')
            for j, col_o in enumerate(os_part_cols)
        ],
        tick=skew_tick,
        prefix='mix_os_col',
    )
  row_inputs = [
      ifthenelse(
          is_os_phase,
          skewed_os_rows[i],
          skewed_ws_rows[i],
          name=f'row_in_mux_{i}',
      )
      for i in range(ARRAY_SIZE)
  ]
  col_inputs = [
      ifthenelse(
          is_os_phase,
          skewed_os_cols[j],
          Stream(ws_part_cols[j] + [0.0] * os_part_h, name=f'ws_col_in_{j}'),
          name=f'col_in_mux_{j}',
      )
      for j in range(ARRAY_SIZE)
  ]
  out = systolic_array(
      row_inputs,
      col_inputs,
      initial_mode=SystolicMode.WEIGHT_STATIONARY,
      switch_mode=Stream(switch_seq, name='switch_mode'),
  )
  return out.deskewed_outputs, out.mode


combined_deskewed_tr, combined_mode_tr = Scheduler.run(
    combined_mode_switch_circuit,
    horizon=combined_horizon,
)
g_combined = Scheduler.graph

combined_mats = extract_matrices(combined_deskewed_tr)
assert combined_mats == [
    reference_matmul_bf16_fp32(a0, weights_epoch0),
    reference_matmul_bf16_fp32(a1, b1),
]
print(f'Stage-1 primitive nodes in combined graph: {len(g_combined.nodes)}')

visualize_graph(
    g_combined,
    title='Stage-1 SYNC Combined WS -> OS Mode Switch',
    height_px=680,
)